# uv and the three Python environments

**Time:** 45–60 minutes  
**Prerequisites:** Notebooks 00–01. Run on Brev or your dev machine.

## Learning objectives

- Install/sync this tutorial project with **uv**
- Register a Jupyter kernel
- Explain why **host**, **Isaac**, and **Cosmos** must stay separate
- Run Brev preflight checks (GPU, disk, env vars)

## Checkpoint

`uv sync` succeeds and you can print which interpreter runs this notebook vs `fanuc_er4ia_sim/.venv`.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "tutorials" / "notebooks"
ROOT = NOTEBOOK_DIR.parent.parent
SIM = ROOT / "fanuc_er4ia_sim"
print("Notebook dir:", NOTEBOOK_DIR)
print("This kernel Python:", sys.executable)

## uv sync (tutorial environment)

In [ ]:
import shutil

print("uv on PATH:", shutil.which("uv"))
# From a terminal in tutorials/notebooks:
#   uv sync
#   uv run python -m ipykernel install --user --name rockwell-tutorials

## Host bootstrap (simulation planning/tests)

The FANUC starter does **not** use uv for its main venv today; it uses `bootstrap.sh`:

```bash
cd fanuc_er4ia_sim
./bootstrap.sh
```

That creates `.venv`, installs `requirements.txt`, generates textures, and runs 33 unit tests.

In [ ]:
host_venv = SIM / ".venv"
if os.name == "nt":
    host_py = host_venv / "Scripts" / "python.exe"
else:
    host_py = host_venv / "bin" / "python"
print("Host venv exists:", host_py.is_file())
print("Expected host python:", host_py)

## Three interpreters

| Variable / path | Runs |
|-----------------|------|
| `fanuc_er4ia_sim/.venv/bin/python` | `plan_sort.py`, tests, `encode_video.py` |
| `$ISAAC/python.sh` | Isaac Sim scripts |
| `$COSMOS` | `perceive.py` (Cosmos3-Nano) |

**Rule:** Never `pip install` Isaac or Cosmos packages into the host venv and expect sim imports to work.

In [ ]:
for key in ("ISAAC", "COSMOS", "COSMOS3_NANO_PATH", "COSMOS3_ROOT", "FANUC_ROBOT_USD"):
    val = os.environ.get(key)
    print(f"{key}={'(set)' if val else '(not set)'}")

## Brev preflight

In [ ]:
def run(cmd: list[str]) -> str:
    try:
        return subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT, timeout=30)
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        return str(exc)

print("--- nvidia-smi (first lines) ---")
out = run(["nvidia-smi", "-L"])
print("\n".join(out.splitlines()[:3]))

print("--- disk ---")
if shutil.which("df"):
    print(run(["df", "-h", "."]).splitlines()[-1])
else:
    du = shutil.disk_usage(ROOT)
    print(f"Free GB: {du.free / 1e9:.1f}")

## Cosmos deps via uv (optional)

If your Cosmos env uses uv for pip operations:

```bash
uv pip install --python "$COSMOS" -r fanuc_er4ia_sim/requirements-cosmos.txt
```

PyTorch and `transformers-cosmos3` are installed separately per `SETUP.md`.

## Exercises

1. Run `uv run python -c "import jupyter; print('ok')"` from `tutorials/notebooks`.
2. From `fanuc_er4ia_sim`, run the host test command from `AGENTS.md`.
3. Write a one-line shell alias or script that exports `ISAAC`, `COSMOS`, and `COSMOS3_NANO_PATH` on Brev startup.

## Troubleshooting

| Issue | Action |
|-------|--------|
| Wrong kernel | Select **Rockwell tutorials** in Jupyter/VS Code |
| Host tests fail | Use Python 3.12 for `fanuc_er4ia_sim/.venv` |
| CUDA false in Cosmos | Check Brev GPU type and `nvidia-smi` |

## Cleanup (Brev)

**Stop or pause** your Brev instance when finished to save credits.